# NB18 · Pythonic Idioms & Pattern Matching

`intermediate/` · notebook 18 of 32 · 34 questions + a mini-project

You can already make Python do almost anything. This notebook is about
writing it the way experienced Python developers do, and about the
features that make that possible.

- **Names and objects.** A variable is a *name* stuck on an object, not a
  box holding a value. Two names can share one object, and that explains
  most "how did *that* change?" bugs.
- **Unpacking with `*` and `**`**, in assignments, calls and literals.
- **The walrus operator `:=`**, which assigns *and* gives back the value in
  the middle of an expression.
- **EAFP** ("easier to ask forgiveness than permission"), **sentinel
  objects**, and the `else` clause on `for`, `while` and `try`.
- **Structural pattern matching.** In NB03 `match` compared against
  literals. Here it takes data apart: sequences, dicts and objects, with
  captures and guards.
- **Style:** PEP 8 (layout and naming), PEP 257 (docstrings) and the Zen of
  Python. Run `import this` in a spare cell to read the Zen; "Readability
  counts" is the line this whole notebook leans on.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB09 to NB17

## What you'll practise

- Names vs objects in depth: aliasing, mutability, is vs ==, caveats about interning
- Extended unpacking (first, *middle, last), the star in calls and literals
- The walrus operator :=
- EAFP idioms, sentinel objects, the else clause on for/while/try
- Advanced match: sequence, mapping and class patterns, captures, guards, nested patterns
- PEP 8, PEP 257, the Zen of Python; refactoring for readability

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB18-Q01 · L1 · Names and objects · Predict

Two names, one list. Predict the output.

**What does this print?**

```python
basket = ["apple", "bread"]
shared = basket
shared.append("milk")
print(basket)
print(basket is shared)
```

<details><summary><b>Hint</b></summary>

`shared = basket` doesn't copy anything. Ask: how many lists exist?

</details>

<details><summary><b>Expected output</b></summary>

```text
['apple', 'bread', 'milk']
True
```

</details>

<details><summary><b>Solution</b></summary>

There is only **one** list. `shared = basket` sticks a second name on the same object, so appending through either name changes the one list both point to. `is` confirms they are the same object.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
basket = ["apple", "bread"]
shared = basket.copy()
shared.append("milk")
print(basket)
print(basket is shared)
```

When a second, independent list is wanted, seniors say so with `.copy()` (or `list(basket)`). Assignment never copies in Python, and knowing that is the key to every aliasing bug in this notebook.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB18-Q02 · L1 · Extended unpacking · Write

With **one** assignment, put the first lap time in `first`, the last in
`last` and everything in between in `middle`. Print the three:

```text
62.1 [59.8, 60.4, 61.0] 58.9
```

**Starting code (already in the cell below):**

```python
lap_times = [62.1, 59.8, 60.4, 61.0, 58.9]
```

<details><summary><b>Hint</b></summary>

A starred name on the left of `=` collects "everything else" into a list.

</details>

<details><summary><b>Expected output</b></summary>

```text
62.1 [59.8, 60.4, 61.0] 58.9
```

</details>

<details><summary><b>Solution</b></summary>

```python
first, *middle, last = lap_times
print(first, middle, last)
```

Python fills `first` and `last` from the ends, and the starred name `*middle` collects whatever is left over, always as a **list**. Only one starred name is allowed per assignment, otherwise Python couldn't know how to split the leftovers.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
first, *middle, last = lap_times
print(first, middle, last)
```

Same code. Compare the slicing version: `lap_times[0]`, `lap_times[1:-1]`, `lap_times[-1]`. Three expressions and three chances for an off-by-one, against one line that shows the *shape* of the data.

</details>

In [ ]:
lap_times = [62.1, 59.8, 60.4, 61.0, 58.9]

# your code here


### NB18-Q03 · L1 · Star in calls · Write

Print the guests on one line separated by `, `, using **one** `print()` call and no `join`.

```text
Ada, Linus, Grace
```

**Starting code (already in the cell below):**

```python
guests = ["Ada", "Linus", "Grace"]
```

<details><summary><b>Hint</b></summary>

A `*` in front of a list inside a function call spreads its items out as separate arguments. `print()` has a `sep=` option (NB00).

</details>

<details><summary><b>Expected output</b></summary>

```text
Ada, Linus, Grace
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(*guests, sep=", ")
```

`print(*guests, sep=", ")` is the same as writing `print("Ada", "Linus", "Grace", sep=", ")`. The `*` unpacks the list into three separate arguments.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print(", ".join(guests))
```

`join` is the usual choice for text, but it only accepts strings. `print(*items, sep=...)` works with numbers or any mix of values, because `print` converts each one with `str()` itself.

</details>

In [ ]:
guests = ["Ada", "Linus", "Grace"]

# your code here


### NB18-Q04 · L1 · Star in literals · Write

Build `menu` as one list: the `morning` items, then the `evening` items,
then `"tea"`. Then build `settings` from `defaults` with the `user`'s
choices on top. Use `*` and `**` inside the brackets. Print both:

```text
['eggs', 'toast', 'soup', 'bread', 'tea']
{'theme': 'dark', 'font_size': 14, 'autosave': True}
```

**Starting code (already in the cell below):**

```python
morning = ["eggs", "toast"]
evening = ("soup", "bread")
defaults = {"theme": "light", "font_size": 12, "autosave": True}
user = {"theme": "dark", "font_size": 14}
```

<details><summary><b>Hint</b></summary>

`[*a, *b]` spreads two collections into a new list, whatever kind they are. `{**a, **b}` does the same for dicts; when a key appears twice, the later one wins.

</details>

<details><summary><b>Expected output</b></summary>

```text
['eggs', 'toast', 'soup', 'bread', 'tea']
{'theme': 'dark', 'font_size': 14, 'autosave': True}
```

</details>

<details><summary><b>Solution</b></summary>

```python
menu = [*morning, *evening, "tea"]
settings = {**defaults, **user}
print(menu)
print(settings)
```

`*` inside a list literal unpacks any iterable, so the tuple `evening` joins the list without converting it first. In the dict, `user` comes second, so its values replace the defaults while `autosave` keeps its default. Keys stay in first-seen order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
menu = [*morning, *evening, "tea"]
settings = defaults | user
print(menu)
print(settings)
```

For two dicts, the merge operator `|` (NB06) reads more directly. `{**a, **b}` is still the tool when you mix in extra keys: `{**defaults, **user, "version": 2}`. Both build a **new** dict and leave `defaults` untouched.

</details>

In [ ]:
morning = ["eggs", "toast"]
evening = ("soup", "bread")
defaults = {"theme": "light", "font_size": 12, "autosave": True}
user = {"theme": "dark", "font_size": 14}

# your code here


### NB18-Q05 · L1 · is vs == · Predict

Equal, or the same object? Predict the output.

**What does this print?**

```python
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b, a is b)
result = None
print(result is None)
```

<details><summary><b>Hint</b></summary>

`==` compares values. `is` asks whether two names point at the very same object.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
True
```

</details>

<details><summary><b>Solution</b></summary>

`a` and `b` hold equal values, so `==` is `True`, but they are two separate lists, so `is` is `False`. There is only ever **one** `None` object, so `is None` is the right check.

</details>

<details><summary><b>Senior dev solution</b></summary>

The PEP 8 rule: use `is` only for singletons like `None` (and `True`, `False`, sentinels from Q18). For everything else, compare values with `==`. Q15 shows why `is` on numbers and strings is a trap.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB18-Q06 · L2 · Extended unpacking · Predict

Starred unpacking on a string and on a one-item list. Predict the output.

**What does this print?**

```python
first, *mid, last = "python"
print(first, mid, last)
head, *tail = [7]
print(head, tail)
```

<details><summary><b>Hint</b></summary>

A string is a sequence of characters. The starred name always gets a list, even when there is nothing left for it.

</details>

<details><summary><b>Expected output</b></summary>

```text
p ['y', 't', 'h', 'o'] n
7 []
```

</details>

<details><summary><b>Solution</b></summary>

`"python"` unpacks letter by letter: `p` and `n` go to the ends and the middle four letters become a **list**, not a string. With `[7]`, `head` takes the only item and `tail` gets an empty list `[]` rather than an error.

</details>

<details><summary><b>Senior dev solution</b></summary>

`head, *tail = items` is a common way to split "the first one" from "the rest", and it's safe on a one-item list. On an empty list it raises `ValueError`, so seniors check for empty input first.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB18-Q07 · L2 · Walrus · Write

If `comment` has more than 3 exclamation marks, print `Too excited: N`
(N is the count). Otherwise print `OK`. Count only once, and use `:=` to
name the count inside the `if`.

```text
Too excited: 6
```

**Starting code (already in the cell below):**

```python
comment = "This product is great!!! Totally worth it!!!"
```

<details><summary><b>Hint</b></summary>

`if (name := expression) > 3:` assigns the value to `name` and then compares it. The brackets are needed.

</details>

<details><summary><b>Expected output</b></summary>

```text
Too excited: 6
```

</details>

<details><summary><b>Solution</b></summary>

```python
if (count := comment.count("!")) > 3:
    print(f"Too excited: {count}")
else:
    print("OK")
```

`:=` assigns **and** returns the value, so the count is computed once, compared with 3, and still available as `count` inside the block. Without the brackets, `count := comment.count("!") > 3` would store the *comparison* (`True`) in `count`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
count = comment.count("!")
if count > 3:
    print(f"Too excited: {count}")
else:
    print("OK")
```

In a plain `if`, many seniors still prefer a separate line: it's just as short and easier to scan. The walrus earns its place in `while` loops (Q16) and comprehensions (Q27), where there's no clean line to put the assignment on.

</details>

In [ ]:
comment = "This product is great!!! Totally worth it!!!"

# your code here


### NB18-Q08 · L2 · None checks · Fix

`apple` is a free sample with a price of `0`, but this code says it's not
found. Fix it so it prints:

```text
apple 0
pear 1.25
kiwi not found
```

**Buggy code (copied into the cell below):**

```python
prices = {"apple": 0, "pear": 1.25}
for item in ["apple", "pear", "kiwi"]:
    price = prices.get(item)
    if not price:
        print(item, "not found")
    else:
        print(item, price)
```

<details><summary><b>Hint</b></summary>

`.get()` returns `None` for a missing key. Which *other* value is also falsy?

</details>

<details><summary><b>Expected output</b></summary>

```text
apple 0
pear 1.25
kiwi not found
```

</details>

<details><summary><b>Solution</b></summary>

```python
prices = {"apple": 0, "pear": 1.25}
for item in ["apple", "pear", "kiwi"]:
    price = prices.get(item)
    if price is None:
        print(item, "not found")
    else:
        print(item, price)
```

`not price` is `True` for `None` **and** for `0`, so a real price of zero looks missing. `price is None` asks the exact question: did `.get()` find nothing?

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
prices = {"apple": 0, "pear": 1.25}
for item in ["apple", "pear", "kiwi"]:
    try:
        price = prices[item]
    except KeyError:
        print(item, "not found")
    else:
        print(item, price)
```

The EAFP version doesn't need a special "missing" value at all: it asks for the key and handles the `KeyError`. That also works when `None` is itself a legitimate stored value, which `.get()` can't tell apart.

</details>

In [ ]:
prices = {"apple": 0, "pear": 1.25}
for item in ["apple", "pear", "kiwi"]:
    price = prices.get(item)
    if not price:
        print(item, "not found")
    else:
        print(item, price)


### NB18-Q09 · L2 · for / else · Write

For each day, print the first damaged order, or `all good` if none is
damaged. Use `for` / `else` with `break`, and no flag variable.

```text
Monday: first damaged C88
Tuesday: all good
```

**Starting code (already in the cell below):**

```python
batches = {
    "Monday": [("A17", "shipped"), ("C88", "damaged"), ("D41", "damaged")],
    "Tuesday": [("E02", "shipped"), ("F19", "shipped")],
}
```

<details><summary><b>Hint</b></summary>

The `else` of a `for` loop runs only when the loop finished **without** a `break`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Monday: first damaged C88
Tuesday: all good
```

</details>

<details><summary><b>Solution</b></summary>

```python
for day, orders in batches.items():
    for order_id, status in orders:
        if status == "damaged":
            print(f"{day}: first damaged {order_id}")
            break
    else:
        print(f"{day}: all good")
```

`break` stops at the first damaged order, which also skips the `else`. On Tuesday the loop runs to the end without breaking, so the `else` runs. Read `for ... else` as "for ... and if nothing broke out".

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for day, orders in batches.items():
    damaged = next((oid for oid, status in orders if status == "damaged"), None)
    print(f"{day}: first damaged {damaged}" if damaged else f"{day}: all good")
```

`next()` with a default (NB22) finds the first match in one line. The loop `else` is still the clearer choice when the loop body does real work, but many developers misread it, so seniors keep such loops short.

</details>

In [ ]:
batches = {
    "Monday": [("A17", "shipped"), ("C88", "damaged"), ("D41", "damaged")],
    "Tuesday": [("E02", "shipped"), ("F19", "shipped")],
}

# your code here


### NB18-Q10 · L2 · EAFP with try / else · Write

Convert each reading to a float. Keep the good ones and count the bad
ones. Use `try` / `except ValueError` / `else`, with the `append` in the
`else` block.

```text
[21.5, 22.0, 23.25]
bad: 2
```

**Starting code (already in the cell below):**

```python
readings = ["21.5", "22.0", "error", "23.25", ""]
```

<details><summary><b>Hint</b></summary>

Put only the line that can fail inside `try`. The `else` block runs when `try` raised nothing.

</details>

<details><summary><b>Expected output</b></summary>

```text
[21.5, 22.0, 23.25]
bad: 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
good = []
bad = 0
for text in readings:
    try:
        value = float(text)
    except ValueError:
        bad += 1
    else:
        good.append(value)
print(good)
print("bad:", bad)
```

Trying the conversion and handling failure is EAFP. The alternative, checking first (LBYL), would need a rule for what a valid float looks like, and `"-1.5e3"` or `" 7 "` make that surprisingly hard. `else` keeps the `try` block to the one risky line.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
good = []
bad = 0
for text in readings:
    try:
        value = float(text)
    except ValueError:
        bad += 1
    else:
        good.append(value)
print(good)
print("bad:", bad)
```

Same code. The senior habit is the *small* `try` block: if `append` were inside `try` and somehow raised `ValueError`, the reading would be counted as bad and the real bug hidden.

</details>

In [ ]:
readings = ["21.5", "22.0", "error", "23.25", ""]

# your code here


### NB18-Q11 · L2 · Unpacking · Error

Which error does this raise?

**Which error does this raise?**

```python
name, price = "Coffee", 3.5, "large"
print(name, price)
```

<details><summary><b>Hint</b></summary>

Count the names on the left and the values on the right.

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
```

</details>

<details><summary><b>Solution</b></summary>

`ValueError`: the right side is a tuple of **three** values but there are only two names, so Python reports "too many values to unpack". Unpacking is strict about counts unless a starred name absorbs the extras.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
name, price, *_ = "Coffee", 3.5, "large"
print(name, price)
```

`*_` says "there may be more, and I don't need it". `_` is the conventional name for a value you're deliberately ignoring.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB18-Q12 · L2 · Sequence patterns · Write

Respond to each command by matching on `command.split()`:

```text
You walk north.
You see a dark room.
You pick up the lamp.
Go where?
I don't understand 'dance wildly'.
```

**Starting code (already in the cell below):**

```python
commands = ["go north", "look", "take lamp", "go", "dance wildly"]
```

<details><summary><b>Hint</b></summary>

`case ["go", direction]:` matches a two-item list whose first item is `"go"` and captures the second as `direction`. A list with one item needs its own case. `case _:` catches the rest.

</details>

<details><summary><b>Expected output</b></summary>

```text
You walk north.
You see a dark room.
You pick up the lamp.
Go where?
I don't understand 'dance wildly'.
```

</details>

<details><summary><b>Solution</b></summary>

```python
for command in commands:
    match command.split():
        case ["go", direction]:
            print(f"You walk {direction}.")
        case ["go"]:
            print("Go where?")
        case ["look"]:
            print("You see a dark room.")
        case ["take", item]:
            print(f"You pick up the {item}.")
        case _:
            print(f"I don't understand '{command}'.")
```

A sequence pattern checks the **length** and each position. Literal strings must be equal; bare names capture whatever is there. `["go", direction]` doesn't match `["go"]` because the lengths differ, which is why `go` alone falls to the next case.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for command in commands:
    match command.split():
        case ["go", direction]:
            print(f"You walk {direction}.")
        case ["go"]:
            print("Go where?")
        case ["look"]:
            print("You see a dark room.")
        case ["take", *words] if words:
            print(f"You pick up the {' '.join(words)}.")
        case _:
            print(f"I don't understand '{command}'.")
```

`*words` in a pattern works like starred unpacking, so `take rusty key` is handled too. The guard `if words` keeps a bare `take` out. Seniors pattern-match on the *shape* of the input rather than writing `len(parts) == 2 and parts[0] == "go"` by hand.

</details>

In [ ]:
commands = ["go north", "look", "take lamp", "go", "dance wildly"]

# your code here


### NB18-Q13 · L2 · PEP 8 and PEP 257 · Refactor

This works, but breaks most of PEP 8. Rename everything in `snake_case` with meaningful names, fix the spacing, use `+=`, and add a one-line docstring (PEP 257).

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def CalcAvg(L):
    Sum=0
    for X in L:
        Sum=Sum+X
    return Sum/len(L)
Scores=[80,90,70]
print(CalcAvg(Scores))
```

<details><summary><b>Hint</b></summary>

Functions and variables use `snake_case`. Put spaces around `=` and binary operators, and two blank lines after a top-level function. A docstring is a string on the first line of the function saying what it returns.

</details>

<details><summary><b>Expected output</b></summary>

```text
80.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
def average(scores):
    """Return the mean of a non-empty list of numbers."""
    total = 0
    for score in scores:
        total += score
    return total / len(scores)


exam_scores = [80, 90, 70]
print(average(exam_scores))
```

PEP 8 is Python's style guide: `snake_case` names, spaces around operators, blank lines between top-level definitions. PEP 257 says a docstring is a one-line summary in the imperative ("Return..."). `L` and `X` told the reader nothing; `scores` and `score` say exactly what they hold.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def average(scores):
    """Return the mean of a non-empty list of numbers."""
    return sum(scores) / len(scores)


exam_scores = [80, 90, 70]
print(average(exam_scores))
```

`sum()` replaces the whole loop. The original author probably wrote `Sum` because `sum` was "taken": naming a variable `sum` would hide the built-in. In practice seniors use `statistics.mean` (NB15) and let a formatter such as Black apply PEP 8 automatically.

</details>

In [ ]:
def CalcAvg(L):
    Sum=0
    for X in L:
        Sum=Sum+X
    return Sum/len(L)
Scores=[80,90,70]
print(CalcAvg(Scores))


### NB18-Q14 · L2 · Nested unpacking · Write

Print each stop with its coordinates. Unpack the name and **both**
coordinates right in the `for` line.

```text
Depot: x=0 y=0
Bakery: x=3 y=4
Market: x=-2 y=5
```

**Starting code (already in the cell below):**

```python
stops = {"Depot": (0, 0), "Bakery": (3, 4), "Market": (-2, 5)}
```

<details><summary><b>Hint</b></summary>

`.items()` gives `(name, (x, y))` pairs. The loop target can have the same nested shape, with brackets.

</details>

<details><summary><b>Expected output</b></summary>

```text
Depot: x=0 y=0
Bakery: x=3 y=4
Market: x=-2 y=5
```

</details>

<details><summary><b>Solution</b></summary>

```python
for name, (x, y) in stops.items():
    print(f"{name}: x={x} y={y}")
```

The target `name, (x, y)` mirrors the shape of each item, so Python unpacks both levels at once. No `point[0]`, `point[1]` indexing needed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for name, (x, y) in stops.items():
    print(f"{name}: x={x} y={y}")
```

Same code. Nested unpacking is also a free shape check: if a coordinate ever arrives with three numbers, this line raises `ValueError` straight away instead of silently using the wrong one.

</details>

In [ ]:
stops = {"Depot": (0, 0), "Bakery": (3, 4), "Market": (-2, 5)}

# your code here


---

## L3 · Intermediate

### NB18-Q15 · L3 · Interning · Predict

`is` on numbers and strings built at run time. Predict the output.

**What does this print?**

```python
a = int("256")
b = int("256")
c = int("257")
d = int("257")
print(a is b, c is d, c == d)
word = "hello"
built = "".join(["hel", "lo"])
print(word == built, word is built)
```

<details><summary><b>Hint</b></summary>

CPython keeps one shared copy of each small integer, from -5 to 256. Larger numbers and strings built while the program runs are usually new objects.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False True
True False
```

</details>

<details><summary><b>Solution</b></summary>

`256` is in CPython's small-int cache, so both names get the same object: `True`. `257` isn't cached, so `int("257")` makes two separate objects: `is` is `False` even though `==` is `True`. The string built by `join` equals `"hello"` but is a different object.

</details>

<details><summary><b>Senior dev solution</b></summary>

The cache is an implementation detail of CPython, not a promise of the language, and the results can change with how the code is written. That's exactly why `x is 257` is a bug waiting to happen (recent Pythons even warn about `is` with a literal). Compare values with `==`; keep `is` for `None` and sentinels.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB18-Q16 · L3 · Walrus in while · Write

Read `stream` 5 characters at a time with `stream.read(5)`, printing each
chunk and its length. `read` returns `""` when nothing is left. Use a
`while` loop with `:=` so `read` is written only **once**.

```text
ABCDE 5
FGHIJ 5
KLMNO 5
PQRST 5
UVW 3
```

**Starting code (already in the cell below):**

```python
import io

stream = io.StringIO("ABCDEFGHIJKLMNOPQRSTUVW")
```

<details><summary><b>Hint</b></summary>

`while chunk := stream.read(5):` reads, names the result, and stops when the result is the empty (falsy) string.

</details>

<details><summary><b>Expected output</b></summary>

```text
ABCDE 5
FGHIJ 5
KLMNO 5
PQRST 5
UVW 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
while chunk := stream.read(5):
    print(chunk, len(chunk))
```

Each time round, `stream.read(5)` runs, its result is stored in `chunk`, and the loop continues only if `chunk` is truthy. Without the walrus you need `read` twice: once before the loop and once at the end of the body, and forgetting the second one makes an infinite loop.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from functools import partial

for chunk in iter(partial(stream.read, 5), ""):
    print(chunk, len(chunk))
```

`iter(function, sentinel)` (NB22) calls the function until it returns the sentinel value, and `partial` (NB09) fixes the `5`. It's the pre-walrus idiom for reading files in blocks and you'll still see it in older code. The walrus loop is easier to read.

</details>

In [ ]:
import io

stream = io.StringIO("ABCDEFGHIJKLMNOPQRSTUVW")

# your code here


### NB18-Q17 · L3 · Mapping patterns · Write

Handle each JSON-style command with `match` and **mapping patterns**. A
`move` needs whole-number `x` and `y`; a `say` needs a string `text`;
`quit` needs nothing else. Anything else is a bad command.

```text
Moving by (3, -1)
Saying 'hello'
Bad command: {'action': 'move', 'x': 0}
Goodbye
Bad command: {'action': 'say', 'text': 42}
```

**Starting code (already in the cell below):**

```python
commands = [
    {"action": "move", "x": 3, "y": -1},
    {"action": "say", "text": "hello"},
    {"action": "move", "x": 0},
    {"action": "quit"},
    {"action": "say", "text": 42},
]
```

<details><summary><b>Hint</b></summary>

`case {"action": "move", "x": int(x), "y": int(y)}:` needs those keys, checks the values, and captures them. `int(x)` in a pattern means "an `int`, captured as `x`". Extra keys in the dict are allowed.

</details>

<details><summary><b>Expected output</b></summary>

```text
Moving by (3, -1)
Saying 'hello'
Bad command: {'action': 'move', 'x': 0}
Goodbye
Bad command: {'action': 'say', 'text': 42}
```

</details>

<details><summary><b>Solution</b></summary>

```python
for command in commands:
    match command:
        case {"action": "move", "x": int(x), "y": int(y)}:
            print(f"Moving by ({x}, {y})")
        case {"action": "say", "text": str(text)}:
            print(f"Saying {text!r}")
        case {"action": "quit"}:
            print("Goodbye")
        case _:
            print("Bad command:", command)
```

A mapping pattern matches when every listed key is present and its sub-pattern matches. `"move"` is compared by value; `int(x)` checks the type and captures. The second `move` lacks `y`, and the second `say` has a number for `text`, so both fall through to `case _`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def handle(command):
    match command:
        case {"action": "move", "x": int(x), "y": int(y)}:
            return f"Moving by ({x}, {y})"
        case {"action": "say", "text": str(text)}:
            return f"Saying {text!r}"
        case {"action": "quit"}:
            return "Goodbye"
        case _:
            return f"Bad command: {command}"


for command in commands:
    print(handle(command))
```

Seniors put the `match` in a function that *returns* a result, so it can be tested on its own (NB28) and reused for input from a file or a socket. Validation and parsing happen in one place: the pattern is the schema.

</details>

In [ ]:
commands = [
    {"action": "move", "x": 3, "y": -1},
    {"action": "say", "text": "hello"},
    {"action": "move", "x": 0},
    {"action": "quit"},
    {"action": "say", "text": 42},
]

# your code here


### NB18-Q18 · L3 · Sentinel objects · Write

Write `get_setting(settings, key, default=_MISSING)`. It returns the
stored value if `key` exists, otherwise `default` if one was passed, and
otherwise raises `KeyError(key)`. Then add these lines below your
function and run the cell:

```text
print(get_setting(settings, "timeout"))
print(get_setting(settings, "proxy", "none set"))
print(get_setting(settings, "retries", None))
try:
    get_setting(settings, "retries")
except KeyError as error:
    print("KeyError:", error)
```

Expected:

```text
30
None
None
KeyError: 'retries'
```

**Starting code (already in the cell below):**

```python
_MISSING = object()
settings = {"timeout": 30, "proxy": None}
```

<details><summary><b>Hint</b></summary>

`object()` creates a brand-new object that can't be equal to anything a caller could pass. Check `default is _MISSING` to know whether a default was given.

</details>

<details><summary><b>Expected output</b></summary>

```text
30
None
None
KeyError: 'retries'
```

</details>

<details><summary><b>Solution</b></summary>

```python
def get_setting(settings, key, default=_MISSING):
    if key in settings:
        return settings[key]
    if default is _MISSING:
        raise KeyError(key)
    return default


print(get_setting(settings, "timeout"))
print(get_setting(settings, "proxy", "none set"))
print(get_setting(settings, "retries", None))
try:
    get_setting(settings, "retries")
except KeyError as error:
    print("KeyError:", error)
```

If the default were `None`, the function couldn't tell "no default given" from "the caller *wants* `None`". The private `_MISSING` object is a **sentinel**: nobody else has a reference to it, so `is _MISSING` is only true when the caller passed nothing. `proxy` exists with the value `None`, so the stored `None` wins over the default.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def get_setting(settings, key, default=_MISSING):
    try:
        return settings[key]
    except KeyError:
        if default is _MISSING:
            raise
        return default


print(get_setting(settings, "timeout"))
print(get_setting(settings, "proxy", "none set"))
print(get_setting(settings, "retries", None))
try:
    get_setting(settings, "retries")
except KeyError as error:
    print("KeyError:", error)
```

The EAFP version looks the key up once and re-raises the original `KeyError` with a bare `raise` (NB12). The standard library uses this exact sentinel pattern, for example in `dict.pop(key[, default])`.

</details>

In [ ]:
_MISSING = object()
settings = {"timeout": 30, "proxy": None}

# your code here


### NB18-Q19 · L3 · Guards · Write

Describe where each point is, using `match` on the tuple, capture patterns
and a guard:

```text
(0, 0) origin
(4, 0) on the x-axis at 4
(0, -2) on the y-axis at -2
(3, 3) on the diagonal
(2, 5) somewhere else
```

**Starting code (already in the cell below):**

```python
points = [(0, 0), (4, 0), (0, -2), (3, 3), (2, 5)]
```

<details><summary><b>Hint</b></summary>

`case (x, 0):` matches any pair whose second value is 0 and captures the first. A guard `if ...` after a pattern adds an extra condition.

</details>

<details><summary><b>Expected output</b></summary>

```text
(0, 0) origin
(4, 0) on the x-axis at 4
(0, -2) on the y-axis at -2
(3, 3) on the diagonal
(2, 5) somewhere else
```

</details>

<details><summary><b>Solution</b></summary>

```python
for point in points:
    match point:
        case (0, 0):
            print(point, "origin")
        case (x, 0):
            print(point, "on the x-axis at", x)
        case (0, y):
            print(point, "on the y-axis at", y)
        case (x, y) if x == y:
            print(point, "on the diagonal")
        case _:
            print(point, "somewhere else")
```

Cases are tried top to bottom, so `(0, 0)` must come before `(x, 0)`, which would also match it. You can't write `case (x, x)` (a name can be captured only once), so the diagonal needs a guard: the pattern captures, then `if x == y` decides.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def describe(point):
    match point:
        case (0, 0):
            return "origin"
        case (x, 0):
            return f"on the x-axis at {x}"
        case (0, y):
            return f"on the y-axis at {y}"
        case (x, y) if x == y:
            return "on the diagonal"
        case _:
            return "somewhere else"


for point in points:
    print(point, describe(point))
```

Returning text from a function separates deciding from printing. The same `describe` can then label points on a chart or in a test, and each case reads as one rule.

</details>

In [ ]:
points = [(0, 0), (4, 0), (0, -2), (3, 3), (2, 5)]

# your code here


### NB18-Q20 · L3 · Mutability and += · Predict

`+=` on a list and on a tuple, each with a second name. Predict the output.

**What does this print?**

```python
cart = ["tea"]
saved_cart = cart
cart += ["jam"]
order = ("tea",)
saved_order = order
order += ("jam",)
print(cart, saved_cart)
print(order, saved_order)
```

<details><summary><b>Hint</b></summary>

A list can change in place. A tuple can't, so `+=` has to build a new tuple and move the name to it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['tea', 'jam'] ['tea', 'jam']
('tea', 'jam') ('tea',)
```

</details>

<details><summary><b>Solution</b></summary>

For a list, `+=` means *extend in place*, so the one list both names share gets `jam`. A tuple is immutable, so `order += ("jam",)` builds a **new** tuple and points `order` at it; `saved_order` still points at the old one.

</details>

<details><summary><b>Senior dev solution</b></summary>

`x += y` and `x = x + y` behave the same for immutable values but differently for lists. Seniors keep this in mind whenever a list has two names, such as a list passed into a function.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB18-Q21 · L3 · Aliasing · Fix

Booking one seat marks it taken in **both** rows. Fix it so it prints:

```text
['free', 'taken', 'free']
['free', 'free', 'free']
```

**Buggy code (copied into the cell below):**

```python
seats = [["free"] * 3] * 2
seats[0][1] = "taken"
for row in seats:
    print(row)
```

<details><summary><b>Hint</b></summary>

`[row] * 2` doesn't copy `row`. How many inner lists exist?

</details>

<details><summary><b>Expected output</b></summary>

```text
['free', 'taken', 'free']
['free', 'free', 'free']
```

</details>

<details><summary><b>Solution</b></summary>

```python
seats = [["free"] * 3 for _ in range(2)]
seats[0][1] = "taken"
for row in seats:
    print(row)
```

The outer `* 2` repeats a **reference** to the same inner list, so both rows are one object. A comprehension runs `["free"] * 3` once per row, creating separate lists. The inner `* 3` is fine because strings are immutable: sharing them can't cause surprises.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
ROWS, COLUMNS = 2, 3
seats = [["free"] * COLUMNS for _ in range(ROWS)]
seats[0][1] = "taken"
for row in seats:
    print(row)
```

Same fix with named sizes. The rule seniors remember: `* n` is safe for immutable items (numbers, strings, `None`) and a bug for mutable ones (lists, dicts).

</details>

In [ ]:
seats = [["free"] * 3] * 2
seats[0][1] = "taken"
for row in seats:
    print(row)


### NB18-Q22 · L3 · Class patterns · Write

Print the kind and area of each shape using `match` with **class
patterns** and keyword sub-patterns. A `Rect` with equal sides is a
square. Round circle areas to 2 decimal places.

```text
circle 3.14
rectangle 6
square 16
circle 19.63
```

**Starting code (already in the cell below):**

```python
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius


class Rect:
    def __init__(self, width, height):
        self.width = width
        self.height = height


shapes = [Circle(1), Rect(2, 3), Rect(4, 4), Circle(2.5)]
```

<details><summary><b>Hint</b></summary>

`case Circle(radius=r):` matches any `Circle` and captures its `radius` attribute as `r`. Put the square case (with a guard) before the general rectangle.

</details>

<details><summary><b>Expected output</b></summary>

```text
circle 3.14
rectangle 6
square 16
circle 19.63
```

</details>

<details><summary><b>Solution</b></summary>

```python
for shape in shapes:
    match shape:
        case Circle(radius=r):
            print("circle", round(math.pi * r ** 2, 2))
        case Rect(width=w, height=h) if w == h:
            print("square", w * h)
        case Rect(width=w, height=h):
            print("rectangle", w * h)
```

A class pattern first checks `isinstance(shape, Circle)`, then reads the named attributes and matches them against the sub-patterns. `radius=r` captures the attribute. The guarded square case comes first because the general `Rect` case would also match a square.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass
class Circle:
    radius: float


@dataclass
class Rect:
    width: float
    height: float


shapes = [Circle(1), Rect(2, 3), Rect(4, 4), Circle(2.5)]
for shape in shapes:
    match shape:
        case Circle(r):
            print("circle", round(math.pi * r ** 2, 2))
        case Rect(w, h) if w == h:
            print("square", w * h)
        case Rect(w, h):
            print("rectangle", w * h)
```

A `@dataclass` (NB21) writes `__init__` for you and also sets `__match_args__`, which lets patterns use positions like `Rect(w, h)`. Q24 shows what happens when a plain class doesn't have it.

</details>

In [ ]:
import math


class Circle:
    def __init__(self, radius):
        self.radius = radius


class Rect:
    def __init__(self, width, height):
        self.width = width
        self.height = height


shapes = [Circle(1), Rect(2, 3), Rect(4, 4), Circle(2.5)]

# your code here


### NB18-Q23 · L3 · Star in calls · Write

Print a label for each customer by unpacking the dict straight into the
call with `**`:

```text
Ada Lovelace, 12 St James's Sq, SW1Y 4JH London
Linus Torvalds, Mannerheimintie 1, 00100 Helsinki
```

**Starting code (already in the cell below):**

```python
def shipping_label(name, street, city, postcode):
    return f"{name}, {street}, {postcode} {city}"


customers = [
    {"name": "Ada Lovelace", "street": "12 St James's Sq", "city": "London", "postcode": "SW1Y 4JH"},
    {"city": "Helsinki", "postcode": "00100", "name": "Linus Torvalds", "street": "Mannerheimintie 1"},
]
```

<details><summary><b>Hint</b></summary>

`f(**d)` passes each key of `d` as a keyword argument with that name.

</details>

<details><summary><b>Expected output</b></summary>

```text
Ada Lovelace, 12 St James's Sq, SW1Y 4JH London
Linus Torvalds, Mannerheimintie 1, 00100 Helsinki
```

</details>

<details><summary><b>Solution</b></summary>

```python
for customer in customers:
    print(shipping_label(**customer))
```

`shipping_label(**customer)` becomes `shipping_label(name=..., street=..., city=..., postcode=...)`. Because the arguments go by **name**, the order of keys in the dict doesn't matter, as the second customer shows.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
print(*(shipping_label(**customer) for customer in customers), sep="\n")
```

One `print` with a generator and `sep="\n"` works, though the loop is just as good. The real senior point: `**` fails loudly with `TypeError` if a dict has a missing or extra key, which catches bad records instead of printing half a label.

</details>

In [ ]:
def shipping_label(name, street, city, postcode):
    return f"{name}, {street}, {postcode} {city}"


customers = [
    {"name": "Ada Lovelace", "street": "12 St James's Sq", "city": "London", "postcode": "SW1Y 4JH"},
    {"city": "Helsinki", "postcode": "00100", "name": "Linus Torvalds", "street": "Mannerheimintie 1"},
]

# your code here


### NB18-Q24 · L3 · Class patterns · Error

A positional class pattern on a plain class. Which error does this raise?

**Which error does this raise?**

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


match Point(0, 0):
    case Point(0, 0):
        print("origin")
    case _:
        print("elsewhere")
```

<details><summary><b>Hint</b></summary>

`Point(0, 0)` in a `case` is not a call. How would Python know which attribute the first `0` refers to?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: a positional sub-pattern needs the class to say which attribute each position means, through a class attribute `__match_args__`. A plain class doesn't have one, so `Point(0, 0)` as a pattern is an error. `case Point(x=0, y=0):` works without it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Point:
    __match_args__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


match Point(0, 0):
    case Point(0, 0):
        print("origin")
    case _:
        print("elsewhere")
```

`__match_args__ = ("x", "y")` maps positions to attributes. Seniors rarely write it by hand: dataclasses and named tuples (NB15, NB21) set it automatically.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L4 · Somewhat Difficult

### NB18-Q25 · L4 · Capture patterns · Predict

`RED` is meant to be a constant to compare against. Predict the output.

**What does this print?**

```python
RED = "red"
color = "blue"
match color:
    case RED:
        print("It's red!")
print(RED)
```

<details><summary><b>Hint</b></summary>

In a `case`, a bare name is never looked up. It is a *capture* pattern.

</details>

<details><summary><b>Expected output</b></summary>

```text
It's red!
blue
```

</details>

<details><summary><b>Solution</b></summary>

`case RED:` doesn't compare with the variable `RED`. A bare name in a pattern **captures**: it matches anything and assigns the subject to that name. So the case matches `"blue"`, prints `It's red!`, and overwrites `RED` with `"blue"`. (Python refuses to compile it if another case follows, because nothing after it could ever run.)

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Colour:
    RED = "red"


color = "blue"
match color:
    case Colour.RED:
        print("It's red!")
    case _:
        print("Not red")
```

A **dotted** name like `Colour.RED` is a value pattern: it's looked up and compared with `==`. Seniors keep constants in a class or an `Enum` (NB21) for exactly this reason. Literal strings like `case "red":` work too.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB18-Q26 · L4 · Refactoring · Refactor

This works but isn't Pythonic: an index-driven `while` loop, `== True`, `x = x + 1` and string-building with `+ str()`. Rewrite it the Pythonic way.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
temps = [18, 25, 31, 22, 35, 29]
hot_days = 0
i = 0
while i < len(temps):
    if (temps[i] > 28) == True:
        hot_days = hot_days + 1
        print("Day " + str(i + 1) + " is hot: " + str(temps[i]))
    i = i + 1
print("Hot days: " + str(hot_days))
```

<details><summary><b>Hint</b></summary>

`enumerate(..., start=1)` gives day numbers. A comparison is already a `bool`. f-strings (NB02) replace the `+ str()` chains.

</details>

<details><summary><b>Expected output</b></summary>

```text
Day 3 is hot: 31
Day 5 is hot: 35
Day 6 is hot: 29
Hot days: 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
temps = [18, 25, 31, 22, 35, 29]
hot_days = 0
for day, temp in enumerate(temps, start=1):
    if temp > 28:
        hot_days += 1
        print(f"Day {day} is hot: {temp}")
print(f"Hot days: {hot_days}")
```

`enumerate(temps, start=1)` hands you the day number and the value, so there's no index to manage and no way to forget `i = i + 1`. `temp > 28` is already `True` or `False`; comparing it to `True` adds nothing. The f-string shows the output's shape at a glance.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
HOT_THRESHOLD = 28

temps = [18, 25, 31, 22, 35, 29]
hot = [(day, temp) for day, temp in enumerate(temps, start=1) if temp > HOT_THRESHOLD]
for day, temp in hot:
    print(f"Day {day} is hot: {temp}")
print(f"Hot days: {len(hot)}")
```

The magic number `28` gets a name. Collecting the hot days first means the count is `len(hot)`, so it can't drift out of sync with what was printed. "Flat is better than nested" and "Readability counts" from the Zen are the whole brief here.

</details>

In [ ]:
temps = [18, 25, 31, 22, 35, 29]
hot_days = 0
i = 0
while i < len(temps):
    if (temps[i] > 28) == True:
        hot_days = hot_days + 1
        print("Day " + str(i + 1) + " is hot: " + str(temps[i]))
    i = i + 1
print("Hot days: " + str(hot_days))


### NB18-Q27 · L4 · Walrus in comprehensions · Write

Build a list of the valid prices in **one** comprehension that calls
`parse_price` only **once** per item. `0` is a valid price and must be
kept.

```text
[4.5, 3.0, 12.0, 0.0]
```

**Starting code (already in the cell below):**

```python
def parse_price(text):
    """Return the price in text as a float, or None if it isn't a number."""
    try:
        return float(text.strip().removeprefix("$"))
    except ValueError:
        return None


raw = ["$4.50", " 3 ", "free", "$12.00", "N/A", "0"]
```

<details><summary><b>Hint</b></summary>

In the `if` part of a comprehension, `(price := parse_price(text))` stores the result so the expression part can use it. Think about what a plain truthiness test does to `0.0`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[4.5, 3.0, 12.0, 0.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
prices = [price for text in raw if (price := parse_price(text)) is not None]
print(prices)
```

The `if` clause runs first for each item: it calls `parse_price` once, stores the result in `price`, and keeps it only if it isn't `None`. The expression on the left then reuses `price`. The trap: writing `if (price := parse_price(text))` would drop `0.0`, because zero is falsy.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
prices = [price for price in map(parse_price, raw) if price is not None]
print(prices)
```

When the computed value is the thing you keep, `map` (NB10) gives each result a name without any walrus. Seniors reach for `:=` in a comprehension when they need the computed value *and* the original item, e.g. `[(text, p) for text in raw if (p := parse_price(text)) is not None]`.

</details>

In [ ]:
def parse_price(text):
    """Return the price in text as a float, or None if it isn't a number."""
    try:
        return float(text.strip().removeprefix("$"))
    except ValueError:
        return None


raw = ["$4.50", " 3 ", "free", "$12.00", "N/A", "0"]

# your code here


### NB18-Q28 · L4 · Nested patterns · Write

Describe each UI event with one `match`, using nested sequence patterns,
`|` alternatives, `as` captures, class patterns and guards:

```text
click at 10,20
quit
digit 7
key x
scroll up 3
scroll ignored
scroll down 2
resize to 800x600
unknown event ('hover',)
```

Rules: `q` or `Q` means quit. A negative scroll is *up* by that many, a
positive one is *down*, and `0` is ignored. Resize sizes must be whole
numbers.

**Starting code (already in the cell below):**

```python
events = [("click", (10, 20)), ("key", "q"), ("key", "7"), ("key", "x"),
          ("scroll", -3), ("scroll", 0), ("scroll", 2), ("resize", (800, 600)),
          ("hover",)]
```

<details><summary><b>Hint</b></summary>

Patterns nest: `("click", (x, y))`. `"q" | "Q"` matches either. `int() as width` checks the type and captures the value. Put specific cases (like scroll `0`) before general ones.

</details>

<details><summary><b>Expected output</b></summary>

```text
click at 10,20
quit
digit 7
key x
scroll up 3
scroll ignored
scroll down 2
resize to 800x600
unknown event ('hover',)
```

</details>

<details><summary><b>Solution</b></summary>

```python
for event in events:
    match event:
        case ("click", (x, y)):
            print(f"click at {x},{y}")
        case ("key", "q" | "Q"):
            print("quit")
        case ("key", str(char)) if char.isdigit():
            print("digit", char)
        case ("key", char):
            print("key", char)
        case ("scroll", 0):
            print("scroll ignored")
        case ("scroll", int(amount)) if amount < 0:
            print("scroll up", -amount)
        case ("scroll", int(amount)):
            print("scroll down", amount)
        case ("resize", (int() as width, int() as height)):
            print(f"resize to {width}x{height}")
        case _:
            print("unknown event", event)
```

Each pattern describes the whole shape of an event, nested tuples included. `"q" | "Q"` is an *or-pattern*. `int() as width` means "must be an `int`; call it `width`". Order matters: `("scroll", 0)` is tested before the general scroll cases, and the digit case before the general key case, because the later ones would also match.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def describe(event):
    match event:
        case ("click", (x, y)):
            return f"click at {x},{y}"
        case ("key", "q" | "Q"):
            return "quit"
        case ("key", str(char)):
            return f"digit {char}" if char.isdigit() else f"key {char}"
        case ("scroll", 0):
            return "scroll ignored"
        case ("scroll", int(amount)):
            return f"scroll {'up' if amount < 0 else 'down'} {abs(amount)}"
        case ("resize", (int() as width, int() as height)):
            return f"resize to {width}x{height}"
        case _:
            return f"unknown event {event}"


for event in events:
    print(describe(event))
```

A senior keeps one case per *kind* of event and puts small decisions inside it, so the `match` reads like a table of contents. Too many guarded variants of the same shape are a sign the logic belongs in the case body.

</details>

In [ ]:
events = [("click", (10, 20)), ("key", "q"), ("key", "7"), ("key", "x"),
          ("scroll", -3), ("scroll", 0), ("scroll", 2), ("resize", (800, 600)),
          ("hover",)]

# your code here


### NB18-Q29 · L4 · Case order · Fix

`quit` should print `bye`, but it's caught by the wrong case. Fix it so it
prints:

```text
do take with ['lamp']
bye
say something
```

**Buggy code (copied into the cell below):**

```python
def describe(command):
    match command.split():
        case [verb, *rest]:
            return f"do {verb} with {rest}"
        case ["quit"]:
            return "bye"
        case []:
            return "say something"


for text in ["take lamp", "quit", ""]:
    print(describe(text))
```

<details><summary><b>Hint</b></summary>

`match` runs the **first** case that fits. Does `[verb, *rest]` fit `["quit"]`?

</details>

<details><summary><b>Expected output</b></summary>

```text
do take with ['lamp']
bye
say something
```

</details>

<details><summary><b>Solution</b></summary>

```python
def describe(command):
    match command.split():
        case ["quit"]:
            return "bye"
        case [verb, *rest]:
            return f"do {verb} with {rest}"
        case []:
            return "say something"


for text in ["take lamp", "quit", ""]:
    print(describe(text))
```

`[verb, *rest]` matches any list with at least one item, including `["quit"]` (with `rest` empty), so the `quit` case could never run. Specific patterns must come before general ones.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def describe(command):
    match command.split():
        case []:
            return "say something"
        case ["quit"]:
            return "bye"
        case [verb, *rest]:
            return f"do {verb} with {rest}"


for text in ["take lamp", "quit", ""]:
    print(describe(text))
```

Seniors order cases from most specific to most general, top to bottom, so the last case reads as the catch-all. Python only *forbids* an unreachable case after a bare capture or `_`; this kind of shadowing is silent, so it's on you (and a test) to catch it.

</details>

In [ ]:
def describe(command):
    match command.split():
        case [verb, *rest]:
            return f"do {verb} with {rest}"
        case ["quit"]:
            return "bye"
        case []:
            return "say something"


for text in ["take lamp", "quit", ""]:
    print(describe(text))


### NB18-Q30 · L4 · while / else · Write

Try to connect up to `MAX_ATTEMPTS` times. Attempt 1 uses the first
response, attempt 2 the second, and so on. Stop at the first `ok`. Use a
`while` loop with an `else` clause for the "gave up" message.

```text
office: attempt 1 timeout
office: attempt 2 ok
office: connected on attempt 2
train: attempt 1 timeout
train: attempt 2 timeout
train: attempt 3 timeout
train: gave up after 3 attempts
```

**Starting code (already in the cell below):**

```python
MAX_ATTEMPTS = 3
scenarios = {
    "office": ["timeout", "ok", "ok"],
    "train": ["timeout", "timeout", "timeout", "ok"],
}
```

<details><summary><b>Hint</b></summary>

Like `for`, a `while` loop's `else` runs only if the loop ended normally (its condition became false), not through `break`.

</details>

<details><summary><b>Expected output</b></summary>

```text
office: attempt 1 timeout
office: attempt 2 ok
office: connected on attempt 2
train: attempt 1 timeout
train: attempt 2 timeout
train: attempt 3 timeout
train: gave up after 3 attempts
```

</details>

<details><summary><b>Solution</b></summary>

```python
for place, responses in scenarios.items():
    attempt = 0
    while attempt < MAX_ATTEMPTS:
        response = responses[attempt]
        attempt += 1
        print(f"{place}: attempt {attempt} {response}")
        if response == "ok":
            print(f"{place}: connected on attempt {attempt}")
            break
    else:
        print(f"{place}: gave up after {MAX_ATTEMPTS} attempts")
```

In the office, `break` leaves the loop on attempt 2 and skips the `else`. On the train the condition `attempt < 3` becomes false after three timeouts, so the loop ends normally and the `else` runs. The fourth response, `ok`, is never reached.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for place, responses in scenarios.items():
    for attempt, response in enumerate(responses[:MAX_ATTEMPTS], start=1):
        print(f"{place}: attempt {attempt} {response}")
        if response == "ok":
            print(f"{place}: connected on attempt {attempt}")
            break
    else:
        print(f"{place}: gave up after {MAX_ATTEMPTS} attempts")
```

A counted loop is a `for` loop: slicing to `MAX_ATTEMPTS` and `enumerate` replace the hand-managed counter, and `for` / `else` works the same way. Real retry code also waits longer between attempts (exponential backoff, NB31).

</details>

In [ ]:
MAX_ATTEMPTS = 3
scenarios = {
    "office": ["timeout", "ok", "ok"],
    "train": ["timeout", "timeout", "timeout", "ok"],
}

# your code here


### NB18-Q31 · L4 · is None vs == None · Predict

A class that claims to equal everything. Predict the output.

**What does this print?**

```python
class Anything:
    def __eq__(self, other):
        return True


value = Anything()
print(value == None)
print(value is None)
print(value == 42, value == "hello")
```

<details><summary><b>Hint</b></summary>

`==` asks the object (it calls `__eq__`). `is` doesn't ask anybody.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
False
True True
```

</details>

<details><summary><b>Solution</b></summary>

`==` calls the object's `__eq__` method, which can say anything, and this one always says `True`. `is` compares identity directly and can't be overridden, so `value is None` is honestly `False`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is why PEP 8 says "comparisons to singletons like `None` should always be done with `is` or `is not`, never the equality operators". Real libraries (NumPy, SQLAlchemy) do redefine `==`, so `x == None` really can lie.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L5 · Hard

### NB18-Q32 · L5 · Recursive patterns · Write

Evaluate each expression tree. A node is a number, `("neg", x)`, or
`(op, left, right)` with `op` one of `+ - * /`. Use `match` and recursion.
Raise `ValueError` for division by zero, for a `bool` (it is **not** a
valid number here), and for anything else unknown. Print each result, or
`error:` and the message:

```text
14
-15
error: division by zero
error: bad value True
error: unknown node '5'
error: unknown node ('%', 1, 2)
```

**Starting code (already in the cell below):**

```python
expressions = [
    ("+", 2, ("*", 3, 4)),
    ("-", ("neg", 5), 10),
    ("/", ("+", 1, 2), ("-", 4, 4)),
    ("*", 2.5, ("+", 1, True)),
    ("neg", "5"),
    ("%", 1, 2),
]
```

<details><summary><b>Hint</b></summary>

`case int() | float():` matches numbers. `bool` is a subclass of `int`, so `True` matches `int()` too: give `bool()` its own case **first**. Sequence patterns never match strings, so `"5"` falls to the catch-all. Use `{node!r}` in the message.

</details>

<details><summary><b>Expected output</b></summary>

```text
14
-15
error: division by zero
error: bad value True
error: unknown node '5'
error: unknown node ('%', 1, 2)
```

</details>

<details><summary><b>Solution</b></summary>

```python
def evaluate(node):
    match node:
        case bool():
            raise ValueError(f"bad value {node!r}")
        case int() | float():
            return node
        case ("neg", operand):
            return -evaluate(operand)
        case ("+", left, right):
            return evaluate(left) + evaluate(right)
        case ("-", left, right):
            return evaluate(left) - evaluate(right)
        case ("*", left, right):
            return evaluate(left) * evaluate(right)
        case ("/", left, right):
            divisor = evaluate(right)
            if divisor == 0:
                raise ValueError("division by zero")
            return evaluate(left) / divisor
        case _:
            raise ValueError(f"unknown node {node!r}")


for expression in expressions:
    try:
        print(evaluate(expression))
    except ValueError as error:
        print("error:", error)
```

Each case matches one *kind* of node and recurses into its children, so trees of any depth work. Two traps: `True` is an `int`, so without the `bool()` case first it would quietly count as `1`; and `"5"` is not matched by a sequence pattern (strings are deliberately excluded), so it reaches `case _` instead of being taken apart letter by letter.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import operator

OPERATORS = {"+": operator.add, "-": operator.sub, "*": operator.mul}


def evaluate(node):
    match node:
        case bool():
            raise ValueError(f"bad value {node!r}")
        case int() | float():
            return node
        case ("neg", operand):
            return -evaluate(operand)
        case ("/", left, right):
            if (divisor := evaluate(right)) == 0:
                raise ValueError("division by zero")
            return evaluate(left) / divisor
        case (op, left, right) if op in OPERATORS:
            return OPERATORS[op](evaluate(left), evaluate(right))
        case _:
            raise ValueError(f"unknown node {node!r}")


for expression in expressions:
    try:
        print(evaluate(expression))
    except ValueError as error:
        print("error:", error)
```

The three arithmetic cases differ only by operator, so a dispatch table (NB09, with `operator` from NB10) replaces them, and adding `%` becomes a one-entry change. Division keeps its own case because it has an extra rule. This is the core of every interpreter, including Python's own.

</details>

In [ ]:
expressions = [
    ("+", 2, ("*", 3, 4)),
    ("-", ("neg", 5), 10),
    ("/", ("+", 1, 2), ("-", 4, 4)),
    ("*", 2.5, ("+", 1, True)),
    ("neg", "5"),
    ("%", 1, 2),
]

# your code here


### NB18-Q33 · L5 · Mapping patterns · Write

Validate each record. A valid record has a string `name` and a whole-number
`age` of 0 or more (a `bool` is **not** an age). Print valid records with
their extra keys sorted (or `none`), and print anything else as invalid:

```text
Ada (36) extra: city
invalid: {'name': 'Linus', 'age': True}
invalid: {'name': 'Grace', 'age': -1}
Alan (41) extra: city, email
invalid: {'name': 7, 'age': 30}
invalid: {'age': 25}
Joan (0) extra: none
```

**Starting code (already in the cell below):**

```python
records = [
    {"name": "Ada", "age": 36, "city": "London"},
    {"name": "Linus", "age": True},
    {"name": "Grace", "age": -1},
    {"name": "Alan", "age": 41, "email": "alan@example.com", "city": "Wilmslow"},
    {"name": 7, "age": 30},
    {"age": 25},
    {"name": "Joan", "age": 0},
]
```

<details><summary><b>Hint</b></summary>

`**rest` at the end of a mapping pattern captures the keys you didn't name, as a dict. `int(age)` also matches `True`, so reject `bool` ages in an earlier case. `", ".join(...) or "none"` gives a fallback for an empty join.

</details>

<details><summary><b>Expected output</b></summary>

```text
Ada (36) extra: city
invalid: {'name': 'Linus', 'age': True}
invalid: {'name': 'Grace', 'age': -1}
Alan (41) extra: city, email
invalid: {'name': 7, 'age': 30}
invalid: {'age': 25}
Joan (0) extra: none
```

</details>

<details><summary><b>Solution</b></summary>

```python
for record in records:
    match record:
        case {"age": bool()}:
            print("invalid:", record)
        case {"name": str(name), "age": int(age), **rest} if age >= 0:
            extras = ", ".join(sorted(rest)) or "none"
            print(f"{name} ({age}) extra: {extras}")
        case _:
            print("invalid:", record)
```

`**rest` collects every key not named in the pattern, so `sorted(rest)` gives the extra key names. Mapping patterns ignore extra keys, so the first case only needs `"age"` to catch boolean ages before the `int()` check can accept them. The guard `if age >= 0` rejects `-1`, and `age 0` is valid, so don't test `if age`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def check(record):
    match record:
        case {"name": str(name), "age": int(age), **rest} if (
            not isinstance(age, bool) and age >= 0
        ):
            return f"{name} ({age}) extra: {', '.join(sorted(rest)) or 'none'}"
        case _:
            return f"invalid: {record}"


for record in records:
    print(check(record))
```

Putting every rule for a valid record into one guard keeps the valid shape in one place. The `bool`-is-an-`int` trap is real: JSON's `true` becomes `True`, and plenty of validators have accepted it as `1`. For bigger schemas, seniors use dataclasses (NB21) or a validation library.

</details>

In [ ]:
records = [
    {"name": "Ada", "age": 36, "city": "London"},
    {"name": "Linus", "age": True},
    {"name": "Grace", "age": -1},
    {"name": "Alan", "age": 41, "email": "alan@example.com", "city": "Wilmslow"},
    {"name": 7, "age": 30},
    {"age": 25},
    {"name": "Joan", "age": 0},
]

# your code here


### NB18-Q34 · L5 · Mutable defaults · Fix

Each call without a list should start from an empty list, but tags leak
from one call to the next. Fix it so it prints:

```text
['urgent']
['billing']
['existing', 'vip']
['later']
```

**Buggy code (copied into the cell below):**

```python
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags


print(add_tag("urgent"))
print(add_tag("billing"))
print(add_tag("vip", ["existing"]))
print(add_tag("later"))
```

<details><summary><b>Hint</b></summary>

A default value is created **once**, when `def` runs, not on every call. Which object does every call without `tags` share?

</details>

<details><summary><b>Expected output</b></summary>

```text
['urgent']
['billing']
['existing', 'vip']
['later']
```

</details>

<details><summary><b>Solution</b></summary>

```python
def add_tag(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags


print(add_tag("urgent"))
print(add_tag("billing"))
print(add_tag("vip", ["existing"]))
print(add_tag("later"))
```

The default `[]` is one list object stored with the function. Every call that relies on the default appends to that same list, so it keeps growing. Using `None` as the default and creating a fresh list inside the function gives each call its own. Use `is None`, not `tags or []`: the `or` version would throw away an empty list the caller passed in on purpose.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def add_tag(tag, tags=()):
    return [*tags, tag]


print(add_tag("urgent"))
print(add_tag("billing"))
print(add_tag("vip", ["existing"]))
print(add_tag("later"))
```

An immutable default (an empty tuple) can't be changed, so it can't leak. Returning a **new** list with `[*tags, tag]` also stops the function from changing the caller's list behind their back: it's a pure function (NB07). Linters flag `def f(x=[])` for exactly this bug.

</details>

In [ ]:
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags


print(add_tag("urgent"))
print(add_tag("billing"))
print(add_tag("vip", ["existing"]))
print(add_tag("later"))


---

## Mini-project

Pull together everything from this notebook.

## NB18-P · L5 · Mini-project: Text Adventure · Write

Write the command interpreter for a tiny text adventure, driven by
`match`. You start in the `hall` with nothing. For each command, print
`> ` and the command, then the response:

- `look`: `You are in <description>. Exits: <exits, sorted>. Items: <items, or none>.`
- `go <direction>`, or the shortcuts `n`, `s`, `e`, `w`: `You go <direction> to
  the <room>.` or `You can't go <direction>.`
- `take <item>`: `You take the <item>.` or `There is no <item> here.`
- `drop <item>`: `You drop the <item>.` or `You don't have a <item>.`
- `inventory` or `i`: `You carry: <items, sorted>.` or `You carry nothing.`
- `quit`: `Bye!`, then ignore any remaining commands.
- Anything else: `I don't understand '<command>'.`

If the commands run out without a `quit`, print `The story ends here.` (use
`for` / `else`). With the given commands, the output is:

```text
> look
You are in a dusty hall. Exits: east, north. Items: lamp.
> take lamp
You take the lamp.
> go north
You go north to the library.
> take key
You take the key.
> take sword
There is no sword here.
> n
You can't go north.
> s
You go south to the hall.
> e
You go east to the kitchen.
> drop lamp
You drop the lamp.
> look
You are in a cold kitchen. Exits: west. Items: lamp.
> i
You carry: key.
> go up
You can't go up.
> dance
I don't understand 'dance'.
> quit
Bye!
```

**Starting code (already in the cell below):**

```python
rooms = {
    "hall": {"description": "a dusty hall",
             "exits": {"north": "library", "east": "kitchen"}, "items": ["lamp"]},
    "library": {"description": "a silent library",
                "exits": {"south": "hall"}, "items": ["book", "key"]},
    "kitchen": {"description": "a cold kitchen",
                "exits": {"west": "hall"}, "items": []},
}
commands = ["look", "take lamp", "go north", "take key", "take sword", "n", "s",
            "e", "drop lamp", "look", "i", "go up", "dance", "quit", "look"]
```

<details><summary><b>Hint</b></summary>

Split each command into words. Turn a one-letter shortcut into `["go", direction]` first (a small `match` with `("n" | "s" | "e" | "w") as letter` does it). Then one `match` with sequence patterns and guards like `if direction in room["exits"]` handles every command. Keep the current room name and an inventory list as you go, and `break` on `quit`.

</details>

<details><summary><b>Expected output</b></summary>

```text
> look
You are in a dusty hall. Exits: east, north. Items: lamp.
> take lamp
You take the lamp.
> go north
You go north to the library.
> take key
You take the key.
> take sword
There is no sword here.
> n
You can't go north.
> s
You go south to the hall.
> e
You go east to the kitchen.
> drop lamp
You drop the lamp.
> look
You are in a cold kitchen. Exits: west. Items: lamp.
> i
You carry: key.
> go up
You can't go up.
> dance
I don't understand 'dance'.
> quit
Bye!
```

</details>

<details><summary><b>Solution</b></summary>

```python
SHORTCUTS = {"n": "north", "s": "south", "e": "east", "w": "west"}
current = "hall"
inventory = []

for command in commands:
    print(">", command)
    words = command.split()
    match words:
        case [("n" | "s" | "e" | "w") as letter]:
            words = ["go", SHORTCUTS[letter]]
    room = rooms[current]
    match words:
        case ["look"]:
            exits = ", ".join(sorted(room["exits"]))
            items = ", ".join(room["items"]) or "none"
            print(f"You are in {room['description']}. Exits: {exits}. Items: {items}.")
        case ["go", direction] if direction in room["exits"]:
            current = room["exits"][direction]
            print(f"You go {direction} to the {current}.")
        case ["go", direction]:
            print(f"You can't go {direction}.")
        case ["take", item] if item in room["items"]:
            room["items"].remove(item)
            inventory.append(item)
            print(f"You take the {item}.")
        case ["take", item]:
            print(f"There is no {item} here.")
        case ["drop", item] if item in inventory:
            inventory.remove(item)
            room["items"].append(item)
            print(f"You drop the {item}.")
        case ["drop", item]:
            print(f"You don't have a {item}.")
        case ["inventory" | "i"]:
            if inventory:
                print(f"You carry: {', '.join(sorted(inventory))}.")
            else:
                print("You carry nothing.")
        case ["quit"]:
            print("Bye!")
            break
        case _:
            print(f"I don't understand '{command}'.")
else:
    print("The story ends here.")
```

The first `match` normalises shortcuts so the main `match` only has to handle `go`. Each command gets a guarded case for success and an unguarded case with the same shape for failure, so every input lands somewhere. `break` on `quit` skips both the last `look` and the loop's `else`. Watch the trap from Q25: `case ["go", direction]` *captures* `direction`; it doesn't compare with an existing variable.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
SHORTCUTS = {"n": "north", "s": "south", "e": "east", "w": "west"}


def respond(words, state):
    """Apply one command to the game state and return the reply."""
    room = rooms[state["room"]]
    match words:
        case [letter] if letter in SHORTCUTS:
            return respond(["go", SHORTCUTS[letter]], state)
        case ["look"]:
            return (f"You are in {room['description']}. "
                    f"Exits: {', '.join(sorted(room['exits']))}. "
                    f"Items: {', '.join(room['items']) or 'none'}.")
        case ["go", direction] if (target := room["exits"].get(direction)):
            state["room"] = target
            return f"You go {direction} to the {target}."
        case ["go", direction]:
            return f"You can't go {direction}."
        case ["take", item] if item in room["items"]:
            room["items"].remove(item)
            state["inventory"].append(item)
            return f"You take the {item}."
        case ["take", item]:
            return f"There is no {item} here."
        case ["drop", item] if item in state["inventory"]:
            state["inventory"].remove(item)
            room["items"].append(item)
            return f"You drop the {item}."
        case ["drop", item]:
            return f"You don't have a {item}."
        case ["inventory" | "i"]:
            carried = sorted(state["inventory"])
            return f"You carry: {', '.join(carried)}." if carried else "You carry nothing."
        case ["quit"]:
            return None
        case _:
            return f"I don't understand '{' '.join(words)}'."


state = {"room": "hall", "inventory": []}
for command in commands:
    print(">", command)
    if (reply := respond(command.split(), state)) is None:
        print("Bye!")
        break
    print(reply)
else:
    print("The story ends here.")
```

`respond` is a function that returns text, so the game logic can be tested without printing (NB28), and the loop only does input and output. The walrus earns its place twice: in the guard, where `room["exits"].get(direction)` both tests and names the target, and in the loop. `None` acts as a "stop" signal. In NB21 the state dict would become a `Game` dataclass.

</details>

In [ ]:
rooms = {
    "hall": {"description": "a dusty hall",
             "exits": {"north": "library", "east": "kitchen"}, "items": ["lamp"]},
    "library": {"description": "a silent library",
                "exits": {"south": "hall"}, "items": ["book", "key"]},
    "kitchen": {"description": "a cold kitchen",
                "exits": {"west": "hall"}, "items": []},
}
commands = ["look", "take lamp", "go north", "take key", "take sword", "n", "s",
            "e", "drop lamp", "look", "i", "go up", "dance", "quit", "look"]

# your code here


---

## Done

Next up: **NB19 · Data Structures Algorithms** in `intermediate/`.